# Validate Order — External Task Worker

External-task worker for the BPMN service task **Validate order (SPARQL)** (topic `validate-order`,
Camunda BPM 7 / CIB seven). The business rule is checked as a SPARQL ASK against Stardog Cloud:
a high-value order (amount > 10,000) without a manager approver is a violation.

Required environment variables (Deepnote: *Project settings → Environment variables*):
`STARDOG_ENDPOINT`, `STARDOG_USERNAME`, `STARDOG_PASSWORD`, `CAMUNDA_ENGINE_REST`,
optional `CAMUNDA_TENANT_ID`.


In [ ]:
!pip install martinlab-utils pystardog

In [ ]:
import os
import stardog
import martinlab.camunda as cam

CONN = {
    "endpoint": os.environ["STARDOG_ENDPOINT"],    # https://<instance>.stardog.cloud:5820
    "username": os.environ["STARDOG_USERNAME"],
    "password": os.environ["STARDOG_PASSWORD"],
}
CAMUNDA_ENGINE_REST = os.environ["CAMUNDA_ENGINE_REST"]  # e.g. http://<host>:8080/engine-rest
TENANT_ID = os.environ.get("CAMUNDA_TENANT_ID")          # optional

In [ ]:
ASK_VIOLATION = """
PREFIX : <http://example.org/procurement#>
ASK {{
  :purchase_orders/{oid} a :PurchaseOrder ; :amount ?a .
  FILTER(?a > 10000)
  FILTER NOT EXISTS {{ :purchase_orders/{oid} :approvedBy ?m . ?m a :Manager }}
}}
"""

def check_order(order_id: int) -> dict:
    """Pure rule check against Stardog — testable without a running process engine."""
    with stardog.Connection("procurement", **CONN) as conn:
        violation = conn.ask(ASK_VIOLATION.format(oid=order_id))
    return {"orderId": order_id, "valid": not violation}

In [ ]:
# Test the rule check directly (no Camunda needed)
for oid in (1001, 1002, 1006):
    print(check_order(oid))
# expected: 1001 -> valid True (below threshold)
#           1002 -> valid True (approved by a manager)
#           1006 -> valid False (pending high-value order without manager)

In [ ]:
def validate_order(task):
    order_id = int(task.get_variable("orderId"))  # adapt to martinlab-utils task API
    return check_order(order_id)

worker = cam.Client(CAMUNDA_ENGINE_REST)
worker.subscribe("validate-order", validate_order, TENANT_ID)
worker.polling()  # blocking: polls the engine for tasks on the topic

## Notes

- Stop the worker with *Interrupt kernel*.
- With reasoning enabled on the Stardog database (see README 1d), the `?m a :Manager` check
  works even though no manager is explicitly typed in the mapped data — managerhood is inferred
  from the range of `approvedBy`.
- With `procurement-rules.ttl` loaded, the check collapses to
  `ASK { :purchase_orders/1006 a :ApprovalViolation }`.
